In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN
from scipy.spatial import cKDTree
trips_df = pd.read_csv('<PROCESSED_DATA_DIR>/trips_cleaned_wide.csv')
positions_df = pd.read_csv('<PROCESSED_DATA_DIR>/trip_positions_cleaned.csv')
stops_ref_df = pd.read_csv('<CLEANED_DATA_DIR>/stop_locations_cleaned.csv')

In [ ]:
# target_id for the trip you want to inspect
target_trip_id = 	1047043 

# 1. Get the trip metadata
trip_info = trips_df[trips_df['trip_id'] == target_trip_id]

# 2. Get all GPS positions for that specific trip
trip_breadcrumbs = positions_df[positions_df['trip_id'] == target_trip_id]

stops_ref_df = pd.read_csv('<CLEANED_DATA_DIR>/stop_locations_cleaned.csv')

# Display results
print(f"Trip Info for {target_trip_id}:")
display(trip_info.T)

print(f"Stop Location Info")
display(stops_ref_df.head())

print(f"\nFound {len(trip_breadcrumbs)} GPS points for this trip.")
display(trip_breadcrumbs.head())



In [ ]:
target_trip_id = 1054114 # Replace with your trip ID

# 1. Get trip start info from trips_df
trip_row = trips_df[trips_df['trip_id'] == target_trip_id].iloc[0]
trip_start_date = str(trip_row['started_at_date'])
trip_start_time = str(trip_row['started_at_time'])

# 2. Get the first GPS ping for this trip (sorted by timestamp)
trip_pings = positions_df[positions_df['trip_id'] == target_trip_id].sort_values('timestamp')
first_ping = trip_pings.iloc[0]
ping_date = str(first_ping['date'])
ping_time = str(first_ping['time_only'])

# 3. Validation Check
date_match = trip_start_date == ping_date
time_match = trip_start_time == ping_time

print(f"--- Verification for Trip: {target_trip_id} ---")
print(f"Trips DF Start:    {trip_start_date} {trip_start_time}")
print(f"First GPS Ping:    {ping_date} {ping_time}")
print(f"Date Match:        {date_match}")
print(f"Time Match:        {time_match}")

if date_match and time_match:
    print("✅ Success: Data is consistent.")
else:
    print("❌ Discrepancy detected between trip metadata and GPS pings.")


In [ ]:

def calculate_freshness_score(pings, stops_ref_df, interval):
    """
    DBSCAN clustering to detect frozen GPS pings while ignoring legitimate stops.
    Works at the trip level.
    """
    total_pings = len(pings)
    if total_pings == 0:
        return 0.0
    
    # 1. Clustering Setup
    coords = np.radians(pings[['lat', 'lng']].values)
    kms_per_radian = 6371.0088
    # epsilon represents ~80 meters in radians
    epsilon = 0.08 / kms_per_radian 
    # min_samples: at least 3 minutes worth of pings
    min_pings_stationary = max(2, 180 / interval) 
    
    db = DBSCAN(eps=epsilon, min_samples=int(min_pings_stationary), 
                algorithm='ball_tree', metric='haversine').fit(coords)
    
    pings = pings.copy()
    pings['cluster'] = db.labels_

    
    # 2. Stop Validation Setup for fast spatial lookup
    stop_tree = cKDTree(stops_ref_df[['lat', 'lng']].values)
    frozen_pings_count = 0
    unique_clusters = set(db.labels_) - {-1}
    
    # 3. Analyze each stationary cluster
    for cluster_id in unique_clusters:
        cluster_pings = pings[pings['cluster'] == cluster_id]
        center_lat, center_lng = cluster_pings['lat'].mean(), cluster_pings['lng'].mean()
        
        # Check distance to nearest stop (approx logic: 0.001 deg ~ 111m)
        dist, _ = stop_tree.query([center_lat, center_lng])
        is_near_stop = dist < 0.001 
        
        total_cluster_duration = (pd.to_datetime(cluster_pings['timestamp_utc'].max()) - 
                                  pd.to_datetime(cluster_pings['timestamp_utc'].min())).total_seconds()
        
        # Identity Check: True binary software freeze 
        identical = (cluster_pings['lat'].nunique() == 1) and (cluster_pings['lng'].nunique() == 1)
        
        if not is_near_stop:
            if identical:
                # Scenario: Frozen in the middle of a street (Malfunction)
                frozen_pings_count += len(cluster_pings)
            elif total_cluster_duration > 600: 
                # Scenario: Stationary in a non-stop spot for > 10 mins (Traffic/Breakdown)
                frozen_pings_count += len(cluster_pings) * 0.5 
        else:
            # Scenario: Legitimate Stop, but excessive dwell time (> 45 mins)
            if total_cluster_duration > 2700:
                frozen_pings_count += int(len(cluster_pings) * 0.3)

    score = (1 - frozen_pings_count / total_pings) * 100
    return max(0, round(score, 1))



In [ ]:
def calculate_gps_health(trip_id, positions_df, trips_df, stops_ref_df):
    
    trip = trips_df[trips_df['trip_id'] == trip_id].iloc[0]
    pings = positions_df[positions_df['trip_id'] == trip_id]
    
    total_pings = len(pings)

    if total_pings == 0:
        return {
            'trip_id': trip_id,
            'coverage_score': 0, 'continuity_score': 100, 'freshness_score': 0, 
            'validity_score': 0, 'trigger_score': 0, 'overall_health_score': 0
        }
    
    # 1. Coverage Score
    polling = {'samsara': 20, 'osg': 20, 'synovia': 30, 
               'zonar': 60, 'icabbi': 60}
    interval = polling.get(trip['source'], 30) 
    #duration = (pd.to_datetime(trip['ended_at_date'] + ' ' + trip['ended_at_time']) - 
    #           pd.to_datetime(trips_df['started_at_date'] + ' ' + trips_df['started_at_time'])).total_seconds()
    try:
        start_str = f"{trip['started_at_date']} {trip['started_at_time']}"
        end_str = f"{trip['ended_at_date']} {trip['ended_at_time']}"
        duration = (pd.to_datetime(end_str) - pd.to_datetime(start_str)).total_seconds()
    except:
        duration = 0
    
    
    expected = duration / interval
    coverage = min(total_pings / expected, 1.0) * 100 if expected > 0 else 0

    # 2. Continuity Score
    # Penalize for gaps over 5 minutes
    pings_sorted = pings.sort_values('timestamp_utc')
    timestamps = pd.to_datetime(pings_sorted['timestamp_utc'])
    gaps = timestamps.diff().dt.total_seconds()
    large_gaps = (gaps > 300).sum() 
    continuity = max(0, 100 - (large_gaps * 10))

    
    # 3. Freshness Score (Calling Independent Function)
    freshness = calculate_freshness_score(pings, stops_ref_df, interval)

    # 4. Validity Score
    invalid = (
        ~pings['lat'].between(41, 47) | 
        ~pings['lng'].between(-95, -87)
    ).sum()
    validity = (1 - invalid / total_pings) * 100 if total_pings > 0 else 0

    # 5. Trigger Score
    # NOTE: These scores are ASSUMPTIONS based on logical reasoning from the data definition.
    # The exact numeric values need to be validated with [REDACTED] and [REDACTED] ([REDACTED]).
    # Question to ask: "What does each end_trigger value mean operationally?
    # Does 'api' mean manual operator override? Does 'timer' mean scheduled end 
    # regardless of GPS? What does 'no_students' mean exactly — did the vehicle 
    # complete the route with no riders, or was the trip cancelled before starting?"

    end_trigger_scores = {
    'geofence': 100,      # Best case — GPS worked, system auto-detected school arrival
    'api': 90,            # Assumption — possibly manual override by operator, GPS likely ok
    'timer': 70,          # Assumption — trip ended at scheduled time, GPS status unclear
    'no_students': 60,    # Assumption — no riders on trip, unclear if GPS was healthy
                          # NEED TO CLARIFY: does this mean trip was cancelled or completed empty?
    'old_last_point': 20, # GPS froze — last ping was too stale to be reliable
    'no_gps_data': 0,     # Worst case — trip ended explicitly because GPS completely failed
    }
    trigger = end_trigger_scores.get(trip['end_trigger'], 50)

    # Overall Score (weighted average)
    overall = (
        coverage    * 0.25 +
        continuity  * 0.25 +
        freshness   * 0.20 +
        validity    * 0.15 +
        trigger     * 0.15
    )

    return {
        'trip_id': trip_id,
        'coverage_score': round(coverage, 1),
        'continuity_score': round(continuity, 1),
        'freshness_score': round(freshness, 1),
        'validity_score': round(validity, 1),
        'trigger_score': trigger,
        'overall_health_score': round(overall, 1)
    }

In [ ]:
health_records = []
target_trips = trips_df['trip_id'].unique()

print(f"Processing {len(target_trips)} trips with DBSCAN spatial validation...")
for trip_id in target_trips[:]: 
    record = calculate_gps_health(trip_id, positions_df, trips_df, stops_ref_df)
    if record:
        health_records.append(record)

gps_health_df = pd.DataFrame(health_records)
gps_health_df.head(20)


In [ ]:
gps_health_df.to_csv('<PROCESSED_DATA_DIR>/gps_health_summary_v2.csv', index=False)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# 1. Load the data
df = pd.read_csv('<PROCESSED_DATA_DIR>/gps_health_summary_v2.csv')

# 2. Define Categories
def categorize_health(score):
    if score == 0: return 'Absolute Zero (0)'
    if score < 50: return 'Failing (<50)'
    if score <= 80: return 'Warning (50-80)'
    return 'Healthy (>80)'

df['category'] = df['overall_health_score'].apply(categorize_health)

# 3. Create Dashboard
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot A: Overall Health Distribution
sns.histplot(df['overall_health_score'], bins=30, kde=True, ax=axes[0], color='#2E86C1')
axes[0].set_title('Overall Health Score Distribution', fontsize=14)
axes[0].set_xlabel('Health Score')
axes[0].set_ylabel('Number of Trips')

# Plot B: Category Breakdown (Pie Chart)
category_counts = df['category'].value_counts()
colors = ['#28B463', '#F39C12', '#CB4335', '#5D6D7E'] # Green, Orange, Red, Grey
axes[1].pie(category_counts, labels=category_counts.index, autopct='%1.1f%%', 
            colors=colors, startangle=140, explode=[0.05, 0, 0, 0.1])
axes[1].set_title('Trip Health Categories', fontsize=14)

plt.tight_layout()
plt.show()

# 4. Component Analysis (Average scores for non-zero trips)
active_trips = df[df['overall_health_score'] > 0]
components = ['coverage_score', 'continuity_score', 'freshness_score', 'validity_score', 'trigger_score']
comp_avg = active_trips[components].mean().sort_values()

plt.figure(figsize=(10, 5))
sns.barplot(x=comp_avg.values, y=comp_avg.index, palette='viridis')
plt.title('Average Scores per Component (Active Trips Only)', fontsize=14)
plt.xlabel('Average Score')
plt.show()


In [ ]:
import folium
import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN
from scipy.spatial import cKDTree

def create_health_map(trip_id, positions_df, trips_df, stops_ref_df):
    """
    Generates an interactive Folium map for a trip, highlighting:
    - Moving path (Blue)
    - Legitimate Stops (Green Clusters)
    - GPS Freezes (Red Clusters)
    - Reference Stop Locations (Blue Pins)
    """
    # 1. Filter Data
    pings = positions_df[positions_df['trip_id'] == trip_id].copy()
    if len(pings) < 2:
        return None
    
    trip = trips_df[trips_df['trip_id'] == trip_id].iloc[0]
    
    # 2. Run DBSCAN to identify stationary clusters (80m radius)
    coords = np.radians(pings[['lat', 'lng']].values)
    kms_per_radian = 6371.0088
    epsilon = 0.08 / kms_per_radian 
    # Use 60s for visualization threshold to see more detail
    db = DBSCAN(eps=epsilon, min_samples=3, metric='haversine').fit(coords)
    pings['cluster'] = db.labels_
    
    # 3. Setup Spatial Index for Stops
    stop_tree = cKDTree(stops_ref_df[['lat', 'lng']].values)
    
    # Create Map centered on trip
    m = folium.Map(location=[pings['lat'].mean(), pings['lng'].mean()], 
                  zoom_start=14, tiles="cartodbpositron")
    
    # 4. Plot Trip Path
    points = pings.sort_values('timestamp_utc')[['lat', 'lng']].values.tolist()
    folium.PolyLine(points, color="#2E86C1", weight=3, opacity=0.7, tooltip=f"Trip {trip_id}").add_to(m)
    
    # 5. Highlight Clusters
    unique_clusters = set(db.labels_) - {-1}
    for cluster_id in unique_clusters:
        cluster_pings = pings[pings['cluster'] == cluster_id]
        center_lat, center_lng = cluster_pings['lat'].mean(), cluster_pings['lng'].mean()
        
        # Check if near a stop
        dist, stop_idx = stop_tree.query([center_lat, center_lng])
        is_near_stop = dist < 0.001 
        
        # Determine Color & Label
        if is_near_stop:
            color = "#28B463" # Green (Healthy Stop)
            label = "Legitimate Stop"
            stop_info = stops_ref_df.iloc[stop_idx]['address']
        else:
            identical = (cluster_pings['lat'].nunique() == 1)
            color = "#CB4335" # Red (Frozen/Unknown)
            label = "Potential GPS Freeze" if identical else "Unknown Stationary Point"
            stop_info = "None nearby"

        duration = (pd.to_datetime(cluster_pings['timestamp_utc'].max()) - 
                    pd.to_datetime(cluster_pings['timestamp_utc'].min())).total_seconds()
        
        # Add Cluster Circle
        folium.CircleMarker(
            location=[center_lat, center_lng],
            radius=10,
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.6,
            popup=f"<b>{label}</b><br>Duration: {int(duration)}s<br>Nearest Stop: {stop_info}"
        ).add_to(m)

    # 6. Show ALL reference stops in the vicinity
    # Bound the search to the trip's geographic box (with 1km buffer)
    lat_min, lat_max = pings['lat'].min() - 0.01, pings['lat'].max() + 0.01
    lng_min, lng_max = pings['lng'].min() - 0.01, pings['lng'].max() + 0.01
    
    relevant_stops = stops_ref_df[
        stops_ref_df['lat'].between(lat_min, lat_max) & 
        stops_ref_df['lng'].between(lng_min, lng_max)
    ]
    
    for _, stop in relevant_stops.iterrows():
        folium.Marker(
            location=[stop['lat'], stop['lng']],
            icon=folium.Icon(color="blue", icon="info-sign"),
            tooltip=f"Stop: {stop['address']}",
            opacity=0.4
        ).add_to(m)

    return m


In [ ]:
import pandas as pd
import sys
sys.path.append('../')
from src.gps_viz_utils import create_health_map

# 1. Load the results to find our targets
results_df = pd.read_csv('<PROCESSED_DATA_DIR>/gps_health_summary_v2.csv')

# 2. Identify the Best and Worst trips
best_trip_id = results_df.sort_values(['freshness_score', 'coverage_score'], ascending=False).iloc[0]['trip_id']
worst_trip_id = results_df[results_df['freshness_score'] > 0].sort_values('freshness_score').iloc[0]['trip_id']

print(f"Generating maps for:\n- Best: {best_trip_id}\n- Worst: {worst_trip_id}")

# 3. Create and Save the 'Best' Trip Map
m_best = create_health_map(best_trip_id, positions_df, trips_df, stops_ref_df)
if m_best:
    m_best.save('best_trip_map.html')
    print("✅ Saved 'best_trip_map.html'")

# 4. Create and Save the 'Worst' Trip Map
m_worst = create_health_map(worst_trip_id, positions_df, trips_df, stops_ref_df)
if m_worst:
    m_worst.save('worst_trip_map.html')
    print("✅ Saved 'worst_trip_map.html'")



In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Load Data
health_df = pd.read_csv('<PROCESSED_DATA_DIR>/gps_health_summary_v2.csv')
trips_df = pd.read_csv('<PROCESSED_DATA_DIR>/trips_cleaned_wide.csv')

# 2. Filter for Active Trips and Calculate Average Freshness
active_trips = health_df[health_df['coverage_score'] > 0]
avg_freshness = active_trips['freshness_score'].mean()

# 3. Identify Below-Average Trips
below_avg = active_trips[active_trips['freshness_score'] < avg_freshness].copy()

# 4. Merge with Metadata
merged = below_avg.merge(
    trips_df[['trip_id', 'vehicle_id', 'vendor_id']], 
    on='trip_id', 
    how='left'
)

# 5. Aggregate Data
vendor_stats = merged.groupby('vendor_id').size().sort_values(ascending=False).head(10).reset_index(name='trip_count')
vehicle_stats = merged.groupby('vehicle_id').size().sort_values(ascending=False).head(10).reset_index(name='trip_count')

# 6. Visualization
plt.figure(figsize=(15, 6))

# Plot A: Top 10 Problematic Vendors
plt.subplot(1, 2, 1)
sns.barplot(data=vendor_stats, x='vendor_id', y='trip_count', palette='Reds_r')
plt.title('Top 10 Vendors (Trips Below Avg Freshness)', fontsize=13, fontweight='bold')
plt.xlabel('Vendor ID')
plt.ylabel('Number of Problematic Trips')

# Plot B: Top 10 Problematic Vehicles
plt.subplot(1, 2, 2)
sns.barplot(data=vehicle_stats, x='vehicle_id', y='trip_count', palette='Oranges_r')
plt.title('Top 10 Vehicles (Trips Below Avg Freshness)', fontsize=13, fontweight='bold')
plt.xlabel('Vehicle ID')
plt.ylabel('Number of Problematic Trips')

plt.tight_layout()
plt.show()

# Print Statistics
print(f"Average Freshness Threshold: {avg_freshness:.2f}")
print(f"Total problematic trips: {len(merged)}")


In [ ]:
# --- NEXT CELL ---

# Define the categories using the variables from the previous cell
total_trips_count = len(health_df)
no_data_count = len(health_df[health_df['coverage_score'] == 0])
problematic_count = len(merged)  # 'merged' contains the below-avg trips from your previous cell
healthy_count = total_trips_count - no_data_count - problematic_count

# Prepare data for Pie Chart
labels = ['Healthy Trips', 'Below Avg Freshness', 'No GPS Data']
sizes = [healthy_count, problematic_count, no_data_count]
colors = ['#2ECC71', '#E67E22', '#E74C3C']  # Green, Orange, Red
explode = (0.1, 0, 0)  # Highlight the Healthy slice

# Create Plot
plt.figure(figsize=(8, 8))
plt.pie(sizes, explode=explode, labels=labels, autopct='%1.1f%%', 
        shadow=True, startangle=140, colors=colors, wedgeprops={'edgecolor': 'white'})

# Add a circle at the center to create a Donut Chart look
centre_circle = plt.Circle((0,0), 0.70, fc='white')
fig = plt.gcf()
fig.gca().add_artist(centre_circle)

plt.title(f'Overall GPS Fleet Health (Total Trips: {total_trips_count:,})', fontsize=15, fontweight='bold')
plt.axis('equal') 
plt.show()

# Print detailed summary
print(f"Summary Statistics:")
print(f"- Healthy Trips: {healthy_count:,}")
print(f"- Problematic (Below Avg Freshness): {problematic_count:,}")
print(f"- No Data Recorded: {no_data_count:,}")


In [ ]:
# --- DEEP DIVE CELL ---

# 1. Distribution by Vehicle Type (Bus vs. Van vs. Cab)
plt.figure(figsize=(16, 10))

plt.subplot(2, 2, 1)
# Re-merging with more metadata for deep dive
deep_dive_df = below_avg.merge(
    trips_df[['trip_id', 'vehicle_type', 'trip_type', 'date', 'rsi_avg_eta_diff']], 
    on='trip_id', 
    how='left'
)
sns.countplot(data=deep_dive_df, x='vehicle_type', palette='viridis')
plt.title('Problematic Trips by Vehicle Type', fontweight='bold')

# 2. Distribution by Trip Type (To School vs. From School)
plt.subplot(2, 2, 2)
sns.countplot(data=deep_dive_df, x='trip_type', palette='magma')
plt.title('Problematic Trips by Trip Type', fontweight='bold')

# 3. Correlation: Freshness vs. Coverage
plt.subplot(2, 2, 3)
sns.scatterplot(data=deep_dive_df, x='coverage_score', y='freshness_score', alpha=0.5, color='teal')
plt.title('Freshness vs. Coverage Score', fontweight='bold')

# 4. Temporal Trend: Problematic Trips over Time
plt.subplot(2, 2, 4)
deep_dive_df['date'] = pd.to_datetime(deep_dive_df['date'])
daily_problems = deep_dive_df.groupby('date').size()
daily_problems.plot(color='red', linewidth=2)
plt.title('Problematic Trips Over Time (Date)', fontweight='bold')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

# --- Key Metrics Analysis ---
print("--- Deep Dive Insights ---")
print(f"Vehicle Type Breakdown:\n{deep_dive_df['vehicle_type'].value_counts(normalize=True) * 100}")
print(f"\nTrip Type Breakdown:\n{deep_dive_df['trip_type'].value_counts(normalize=True) * 100}")

# Check if low freshness impacts operational delay (ETA diff)
problem_delay = deep_dive_df['rsi_avg_eta_diff'].mean()
overall_delay = trips_df['rsi_avg_eta_diff'].mean()
print(f"\nAvg ETA Delay (Problematic Trips): {problem_delay:.2f} mins")
print(f"Avg ETA Delay (All Trips): {overall_delay:.2f} mins")


In [ ]:
# --- TARGETED INSIGHTS CELL (FIXED) ---

# 1. Vendor Freshness Comparison
# FIX: Added 'vehicle_id' to the merge columns
all_active = health_df[health_df['coverage_score'] > 0].merge(
    trips_df[['trip_id', 'vendor_id', 'vehicle_id']], on='trip_id'
)
vendor_avg_freshness = all_active.groupby('vendor_id')['freshness_score'].mean().sort_values()

# 2. Vendor 62 Vehicle Breakdown
v62_trips = merged[merged['vendor_id'] == 62]
v62_veh_counts = v62_trips.groupby('vehicle_id').size().sort_values(ascending=False).head(5)

# 3. Vehicle 1014 vs its Vendor (29)
v29_trips = all_active[all_active['vendor_id'] == 29]
v1014_avg = v29_trips[v29_trips['vehicle_id'] == 1014]['freshness_score'].mean()
v29_other_avg = v29_trips[v29_trips['vehicle_id'] != 1014]['freshness_score'].mean()

# Create Visuals
plt.figure(figsize=(18, 5))

# Plot 1: Vendor 13 Comparison
plt.subplot(1, 3, 1)
colors = ['#E74C3C' if x == 13 else '#BDC3C7' for x in vendor_avg_freshness.index]
vendor_avg_freshness.plot(kind='bar', color=colors)
plt.title('Avg Freshness by Vendor (Vendor 13 highlighted)', fontweight='bold')
plt.ylabel('Avg Freshness Score')
plt.axhline(avg_freshness, color='black', linestyle='--', alpha=0.5)

# Plot 2: Vendor 62 Problem Vehicles
plt.subplot(1, 3, 2)
v62_colors = ['#D35400' if x in [3431, 2087] else '#3498DB' for x in v62_veh_counts.index]
v62_veh_counts.plot(kind='bar', color=v62_colors)
plt.title('Vendor 62: Top Problem Vehicles', fontweight='bold')
plt.ylabel('Count of Problematic Trips')
plt.xlabel('Vehicle ID')

# Plot 3: Vehicle 1014 vs Vendor 29 Peers
plt.subplot(1, 3, 3)
plt.bar(['Vehicle 1014', 'Other Vendor 29 Veh'], [v1014_avg, v29_other_avg], color=['#C0392B', '#27AE60'])
plt.title('Vehicle 1014 vs. Vendor 29 Peers', fontweight='bold')
plt.ylabel('Avg Freshness Score')
plt.ylim(min(v1014_avg, v29_other_avg) - 5, 100) # Dynamic zoom

plt.tight_layout()
plt.show()

# Precise data output for report
print(f"Vendor 13 Avg Freshness: {vendor_avg_freshness[13]:.2f}")
print(f"Vehicle 1014 Avg Freshness: {v1014_avg:.2f} (Vendor 29 Peers: {v29_other_avg:.2f})")
